# Notebook 13 — QC-aware privacy–utility sensitivity

Combines the N07 preserved-emotion results with N12 corrected ArcFace/AuraFace similarities and visual-QC decisions from the private Google Sheet. Produces aggregate source-cluster bootstrap CIs. **Not executed yet**. `ACCEPT` indicates visual technical plausibility, never proof of anonymization. No re-identification thresholds are calibrated. Keep row-level data on private Google Drive.


In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd, numpy as np, json
from datetime import datetime, timezone
drive.mount('/content/drive')
RUNS=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=RUNS/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
assert N07.is_dir(), 'Frozen N07 run not found'
from google.colab import auth
import google.auth, gspread
from sklearn.utils import check_random_state
auth.authenticate_user()
N12=RUNS/'nb12_v3_biometric_11faafd9beb1_20261013T211442695856Z_bcf37701'
assert N12.is_dir(), 'Frozen N12 run not found'
SHEET_ID='1rXJ3TfY1VKDIfF_bkp7KQztS7tmFQ8xB-F0L4CpLeCE'
out=RUNS/('nb13_qc_sensitivity_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
out.mkdir(parents=True,exist_ok=False)
print(out)


In [ ]:
n07=pd.read_csv(N07/'raw_results.csv',dtype={'source_id':'string','condition':'string'})
n12=pd.read_csv(N12/'raw_corrected_privacy.csv',dtype={'source_id':'string','condition':'string'})
credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/spreadsheets.readonly'])
sheet=gspread.authorize(credentials).open_by_key(SHEET_ID).worksheet('Pair Review')
review=pd.DataFrame(sheet.get_all_records())
keys=['source_id','condition','seed']
assert len(n07)==len(n12)==len(review)==600
for name,x in [('N07',n07),('N12',n12)]:
 assert not x.duplicated(keys).any(),f'Duplicate {name} keys'
review=review.rename(columns={'Source ID':'source_id','Condition':'condition','Seed':'seed','Overall manual decision':'qc','Reviewer notes':'qc_notes'})
review['seed']=pd.to_numeric(review.seed,errors='raise').astype(int)
assert not review.duplicated(keys).any()
keyset=lambda x:set(map(tuple,x[keys].itertuples(index=False,name=None)))
assert keyset(n07)==keyset(n12)==keyset(review)
assert n12.source_id.nunique()==120
assert set(review.qc)=={'ACCEPT','UNCERTAIN'}
assert (n12.pair_valid_arcface.astype(str).str.lower()=='true').all()
assert (n12.pair_valid_auraface.astype(str).str.lower()=='true').all()
print('All 600 pair keys match.')


In [ ]:
utility_cols=['source_id','condition','seed','emotion_preserved','emotion_status']
df=n12.merge(n07[utility_cols],on=keys,validate='one_to_one').merge(review[keys+['qc','qc_notes']],on=keys,validate='one_to_one')
assert len(df)==600 and df[['arcface_cosine','auraface_cosine']].notna().all().all()
assert (df.emotion_status=='success').all()
df['preserved']=df.emotion_preserved.astype(str).str.lower().map({'true':1,'false':0})
assert df.preserved.notna().all()
rng=np.random.default_rng(20261014)
def cluster_ci(part,col,B=2000):
 groups=[v[col].to_numpy(dtype=float) for _,v in part.groupby('source_id')]
 n=len(groups);boots=[]
 for _ in range(B):
  idx=rng.integers(0,n,n)
  boots.append(float(np.mean(np.concatenate([groups[i] for i in idx]))))
 return np.quantile(boots,[.025,.975]).tolist()
rows=[]
for condition,condition_data in df.groupby('condition',sort=True):
 for status in ['ALL','ACCEPT','UNCERTAIN']:
  part=condition_data if status=='ALL' else condition_data[condition_data.qc==status]
  for metric in ['arcface_cosine','auraface_cosine','preserved']:
   lo,hi=cluster_ci(part,metric)
   rows.append({'condition':condition,'qc_stratum':status,'metric':metric,'n_pairs':len(part),'n_sources':part.source_id.nunique(),'mean':part[metric].mean(),'cluster_CI95_low':lo,'cluster_CI95_high':hi})
summary=pd.DataFrame(rows)
summary.to_csv(out/'qc_sensitivity_aggregates.csv',index=False)
display(summary.round(4))


In [ ]:
# Tuning IDs are explicitly unknown until validated from tuning receipts.
TUNING_SOURCE_IDS=[]
status='NOT_RUN_NO_DOCUMENTED_TUNING_IDS'
if TUNING_SOURCE_IDS:
 assert len(TUNING_SOURCE_IDS)==12 and len(set(TUNING_SOURCE_IDS))==12
 assert set(TUNING_SOURCE_IDS).issubset(set(df.source_id))
 excluded=df[~df.source_id.isin(TUNING_SOURCE_IDS)]
 excluded.groupby('condition').agg(n=('source_id','size'),arcface=('arcface_cosine','mean'),auraface=('auraface_cosine','mean'),emotion_preserved=('preserved','mean')).reset_index().to_csv(out/'tuning_exclusion_sensitivity.csv',index=False)
 status='EXECUTED_WITH_DOCUMENTED_TUNING_IDS'
report={'notebook':'13','source_count':int(df.source_id.nunique()),'n_pairs':len(df),'qc_counts':{str(k):int(v) for k,v in df.qc.value_counts().items()},'tuning_exclusion':status,'neutral_prompt':'NOT_MEASURED_HERE','no_reference_image_quality':'NOT_MEASURED_HERE','privacy_threshold':'NOT_CALIBRATED','visual_qc':'FIRST_PASS_SECOND_REVIEW_PENDING'}
(out/'evidence_status.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
